In [0]:
from pyspark.sql import functions as F

In [0]:
def run_post_validation_audit():
    print("=" * 80)
    print("STARTING POST-VALIDATION RECONCILIATION & INTEGRITY AUDIT")
    print("=" * 80)
    
    # -------------------------------------------------------------------------
    # TEST 1: ROW COUNT RECONCILIATION (Conservation of Records)
    # Target: Bronze Count == Silver Valid Count + Quarantine Count
    # -------------------------------------------------------------------------
    entities = [
        ("orders", "raw_olist_orders_dataset", "valid_orders", "quarantine_orders"),
        ("order_items", "raw_olist_order_items_dataset", "valid_order_items", "quarantine_order_items"),
        ("order_payments", "raw_olist_order_payments_dataset", "valid_order_payments", "quarantine_order_payments"),
        ("order_reviews", "raw_olist_order_reviews_dataset", "valid_order_reviews", "quarantine_order_reviews"),
        ("products", "raw_olist_products_dataset", "valid_products", "quarantine_products"),
        ("customers", "raw_olist_customers_dataset", "valid_customers", "quarantine_customers"),
        ("sellers", "raw_olist_sellers_dataset", "valid_sellers", "quarantine_sellers"),
        ("geolocation", "raw_olist_geolocation_dataset", "valid_geolocation", "quarantine_geolocation")
    ]
    
    reconciliation_passed = True
    print("\n--- 1. ROW CONSERVATION AUDIT ---")
    
    for label, bronze_tbl, silver_tbl, quar_tbl in entities:
        bronze_count = spark.table(f"asb.olist_bronze.{bronze_tbl}").count()
        silver_count = spark.table(f"asb.olist_silver.{silver_tbl}").count()
        quar_count = spark.table(f"asb.quarantine.{quar_tbl}").count()
        
        diff = bronze_count - (silver_count + quar_count)
        
        status = "PASSED" if diff == 0 else "FAILED"
        if diff != 0:
            reconciliation_passed = False
            
        print(f"[{status}] {label.upper():<15} | Bronze: {bronze_count:<7} | Valid: {silver_count:<7} | Quarantined: {quar_count:<6} | Delta: {diff}")
        
    assert reconciliation_passed, "FATAL: Record reconciliation failed! Some records were lost or duplicated during the validation split."

    # -------------------------------------------------------------------------
    # TEST 2: LEAKAGE DETECTION IN SILVER (Zero Tolerated Anomalies)
    # Target: Silver tables must return 0 records when checked against DQ rules
    # -------------------------------------------------------------------------
    print("\n--- 2. SILVER PURITY & LEAKAGE AUDIT ---")
    leakage_failures = 0

    # 2a. Check Orders Purity (Dates and Status)
    df_valid_orders = spark.table("asb.olist_silver.valid_orders")
    leaked_orders = df_valid_orders.filter(
        (F.col("customer_id").isNull()) |
        (~F.col("order_status").isin(['delivered', 'shipped', 'canceled', 'invoiced', 'processing', 'created', 'approved', 'unavailable'])) |
        ((F.col("order_delivered_customer_date").isNotNull()) & 
         (F.to_timestamp("order_delivered_customer_date") < F.to_timestamp("order_purchase_timestamp")))
    ).count()
    
    print(f"[{'PASSED' if leaked_orders == 0 else 'FAILED'}] olist_silver.valid_orders anomaly leakage: {leaked_orders}")
    leakage_failures += leaked_orders

    # 2b. Check Order Items Purity (Prices and Freights)
    df_valid_items = spark.table("asb.olist_silver.valid_order_items")
    leaked_items = df_valid_items.filter(
        (F.col("price").cast("double") <= 0.0) |
        (F.col("freight_value").cast("double") < 0.0)
    ).count()
    
    print(f"[{'PASSED' if leaked_items == 0 else 'FAILED'}] olist_silver.valid_order_items anomaly leakage: {leaked_items}")
    leakage_failures += leaked_items

    # 2c. Check Payments Purity (Negative Values)
    df_valid_payments = spark.table("asb.olist_silver.valid_order_payments")
    leaked_payments = df_valid_payments.filter(
        F.col("payment_value").cast("double") <= 0.0
    ).count()
    
    print(f"[{'PASSED' if leaked_payments == 0 else 'FAILED'}] olist_silver.valid_order_payments anomaly leakage: {leaked_payments}")
    leakage_failures += leaked_payments

    # 2d. Check Reviews Purity (Score bounds)
    df_valid_reviews = spark.table("asb.olist_silver.valid_order_reviews")
    leaked_reviews = df_valid_reviews.filter(
        ~F.col("review_score").cast("int").between(1, 5)
    ).count()
    
    print(f"[{'PASSED' if leaked_reviews == 0 else 'FAILED'}] olist_silver.valid_order_reviews anomaly leakage: {leaked_reviews}")
    leakage_failures += leaked_reviews

    # 2e. Check Customers Purity (Unique ID completeness)
    df_valid_customers = spark.table("asb.olist_silver.valid_customers")
    leaked_cust = df_valid_customers.filter(
        F.col("customer_unique_id").isNull() | (F.trim(F.col("customer_unique_id")) == "")
    ).count()
    
    print(f"[{'PASSED' if leaked_cust == 0 else 'FAILED'}] olist_silver.valid_customers anomaly leakage: {leaked_cust}")
    leakage_failures += leaked_cust

    assert leakage_failures == 0, f"FATAL: Detected {leakage_failures} contaminated records inside Silver layer!"

    # -------------------------------------------------------------------------
    # TEST 3: QUARANTINE INTEGRITY (Auditability of Failures)
    # Target: Quarantine tables must have valid error reasons and payloads
    # -------------------------------------------------------------------------
    print("\n--- 3. QUARANTINE METADATA INTEGRITY AUDIT ---")
    quarantine_integrity_passed = True
    
    for label, _, _, quar_tbl in entities:
        df_quar = spark.table(f"asb.quarantine.{quar_tbl}")
        
        # Check if any quarantined row has empty rule arrays or null raw_payload
        invalid_quar_records = df_quar.filter(
            (F.col("_failure_count") <= 0) |
            (F.col("failed_rules").isNull()) |
            (F.size(F.col("failed_rules")) == 0) |
            (F.col("raw_payload").isNull())
        ).count()
        
        status = "PASSED" if invalid_quar_records == 0 else "FAILED"
        if invalid_quar_records > 0:
            quarantine_integrity_passed = False
            
        print(f"[{status}] {quar_tbl:<30} | Unexplainable Quarantine Rows: {invalid_quar_records}")

    assert quarantine_integrity_passed, "FATAL: Some quarantined records lack audit metadata or failing rule IDs!"

    print("\n" + "=" * 80)
    print("ALL DATA QUALITY RECONCILIATION & PURITY TESTS PASSED SUCCESSFULLY!")
    print("=" * 80)

In [0]:
run_post_validation_audit()